<a href="https://colab.research.google.com/github/heyitsme-404/AI-Assisted-Legacy-SQL-Migration-Tool/blob/main/AI_Assisted_Legacy_SQL_Migration_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
#SETUP

import sqlite3                         # database tool
conn = sqlite3.connect(":memory:")     # open a temporary database
cur = conn.cursor()                    # my "pen" for SQL


#EVENTS TABLE

cur.execute("CREATE TABLE events (name TEXT, city TEXT, year INTEGER)")
cur.execute("INSERT INTO events VALUES ('Delhi Fest', 'delhi', 2025)")
cur.execute("INSERT INTO events VALUES ('Coventry Meetup', 'coventry', 2026)")   # no one here
cur.execute("INSERT INTO events VALUES ('London Expo', 'London', 2026)")


#PEOPLE TABLE

cur.execute("CREATE TABLE people (name TEXT, event TEXT)")
cur.execute("INSERT INTO people VALUES ('she', 'Delhi Fest')")
cur.execute("INSERT INTO people VALUES ('me', 'London Expo')")
cur.execute("INSERT INTO people VALUES ('her', 'London Expo')")   # 2 people here


#CHECKING DATA

print(cur.execute("SELECT * FROM events").fetchall())
print(cur.execute("SELECT * FROM people").fetchall())

[('Delhi Fest', 'delhi', 2025), ('Coventry Meetup', 'coventry', 2026), ('London Expo', 'London', 2026)]
[('she', 'Delhi Fest'), ('me', 'London Expo'), ('her', 'London Expo')]


In [7]:
#OLD QUERIES (legacy style)

old_queries = [
    "SELECT p.name, e.city FROM people p, events e WHERE p.event = e.name",   # old join (comma)
    "SELECT * FROM events WHERE city = 'London' AND year = 2026",             # SELECT * (all columns)
    "SELECT name FROM events WHERE name IN (SELECT event FROM people)"        # subquery
]

#p = nickname for people, e = nickname for events.

# run each one and show the result

for q in old_queries:
    print(cur.execute(q).fetchall())

    #Coventry Meetup is missing because nobody goes there.

[('she', 'delhi'), ('me', 'London'), ('her', 'London')]
[('London Expo', 'London', 2026)]
[('Delhi Fest',), ('London Expo',)]


In [8]:
#NEW QUERIES (AI rewrites, prompt A) #CHATGPT

new_queries = [
    """SELECT p.name, e.city
FROM people AS p
INNER JOIN events AS e
    ON p.event = e.name;""",

    """SELECT *
FROM events
WHERE city = 'London'
  AND year = 2026;""",

    """SELECT name
FROM events
WHERE name IN (
    SELECT event
    FROM people
);"""
]


#COMPARE OLD vs NEW

passed = 0                                            # count of passes

for i in range(len(old_queries)):                     # i = 0, 1, 2
    old_result = sorted(cur.execute(old_queries[i]).fetchall())
    new_result = sorted(cur.execute(new_queries[i]).fetchall())

    if old_result == new_result:                      # same results?
        print(f"Query {i+1}: PASS")
        passed = passed + 1
    else:
        print(f"Query {i+1}: FAIL")   #{i+1}	Shows Query 1, 2, 3 instead of 0, 1, 2
        print("  old:", old_result)
        print("  new:", new_result)

print(f"Accuracy: {passed}/{len(old_queries)}")

Query 1: PASS
Query 2: PASS
Query 3: PASS
Accuracy: 3/3


In [9]:
#TEST FUNCTION

def run_tests(old_list, new_list, label):
    print(f"--- {label} ---")    #LABEL - A name to show, like “Prompt A”
    passed = 0
    for i in range(len(old_list)):
        old_result = sorted(cur.execute(old_list[i]).fetchall())
        new_result = sorted(cur.execute(new_list[i]).fetchall())
        if old_result == new_result:
            print(f"Query {i+1}: PASS")
            passed = passed + 1
        else:
            print(f"Query {i+1}: FAIL")
            print("  old:", old_result)
            print("  new:", new_result)
    print(f"Accuracy: {passed}/{len(old_list)}\n")


#PROMPT B (strict)

prompt_b = [
    """SELECT p.name, e.city
FROM people p
INNER JOIN events e
    ON p.event = e.name;""",

    """SELECT name, city, year
FROM events
WHERE city = 'London'
  AND year = 2026;""",

    """SELECT DISTINCT e.name
FROM events e
INNER JOIN people p
    ON e.name = p.event;"""
]


# TRAP: query 3 without DISTINCT (should FAIL)

trap = [old_queries[0], old_queries[1],
        "SELECT e.name FROM events e INNER JOIN people p ON e.name = p.event"]


run_tests(old_queries, new_queries, "Prompt A")
run_tests(old_queries, prompt_b, "Prompt B")
run_tests(old_queries, trap, "Trap test")

--- Prompt A ---
Query 1: PASS
Query 2: PASS
Query 3: PASS
Accuracy: 3/3

--- Prompt B ---
Query 1: PASS
Query 2: PASS
Query 3: PASS
Accuracy: 3/3

--- Trap test ---
Query 1: PASS
Query 2: PASS
Query 3: FAIL
  old: [('Delhi Fest',), ('London Expo',)]
  new: [('Delhi Fest',), ('London Expo',), ('London Expo',)]
Accuracy: 2/3



In [10]:
#ERROR LOG

import csv

log = [
    {"prompt": "A", "query": 1, "result": "PASS", "note": "Changed comma join to INNER JOIN"},
    {"prompt": "A", "query": 2, "result": "PASS", "note": "Only reformatted, kept SELECT *"},
    {"prompt": "A", "query": 3, "result": "PASS", "note": "Only reformatted, kept subquery"},
    {"prompt": "B", "query": 1, "result": "PASS", "note": "INNER JOIN"},
    {"prompt": "B", "query": 2, "result": "PASS", "note": "Named columns instead of SELECT *"},
    {"prompt": "B", "query": 3, "result": "PASS", "note": "Subquery to JOIN, added DISTINCT"},
    {"prompt": "Trap", "query": 3, "result": "FAIL", "note": "JOIN without DISTINCT duplicated London Expo"}
]

with open("test_log.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["prompt", "query", "result", "note"])
    writer.writeheader()
    writer.writerows(log)

print("Log saved")

Log saved
